# GVH–CHH · A1 · Précision de mesure pour deux compartiments de gaz
**CHH-COMP-01 · Cas gaz · Budget d’erreur v0.1 · Brouillon pré-gel**  
Protocole CHH v1.0 · [Notebook de référence avec volumes exacts](./CHH_COMP_01_Deux_Compartiments_Gaz_A1_v01.ipynb)

## Niveau 1 — CHH-STRUCTURE et INVERSE-INFERENCE

Deux compartiments rigides contiennent le même gaz idéal. L’enceinte est fermée aux échanges de matière et la température initiale des deux compartiments et la température finale sont identiques. Pressions absolues en kPa, volumes en litres. On estime la pression initiale du deuxième compartiment à partir des mesures de la pression initiale du premier, de la pression finale et des deux volumes. Le seuil de travail est une RMSE de 1 kPa sur la population simulée. L’étude est statique et utilise des données synthétiques.

## Niveau 2 — KNOWN-PHYSICS et propagation d’incertitude

Posons $w=V_2/(V_1+V_2)$ et $\Delta p=p_2-p_1$. La conservation de la quantité de gaz donne $p_f=p_1+w\Delta p$. L’inversion applique $\widehat p_2=p_1^{\rm mes}+(p_f^{\rm mes}-p_1^{\rm mes})/\widehat w$ aux mesures.

Si $\eta_1,\eta_2$ sont les erreurs relatives des volumes, alors $\delta w\simeq w(1-w)(\eta_2-\eta_1)$. Au premier ordre,
$$\delta p_2\simeq\frac{\epsilon_f-(1-w)\epsilon_1}{w}-(1-w)\Delta p(\eta_2-\eta_1).$$

Avec des erreurs indépendantes et centrées, un même écart-type $\sigma_p$ sur chacune des pressions et un même écart-type relatif $r$ sur chacun des volumes, la RMSE anticipée vaut
$$\mathrm{RMSE}_{\rm lin}=
\sqrt{\left[\frac{\sqrt{1+(1-w)^2}}{w}\sigma_p\right]^2+
\left[\sqrt{2}(1-w)\,D\,r\right]^2},\qquad D=\sqrt{\mathbb E[(\Delta p)^2]}.$$

Ce budget quadratique dépend de l’amplitude du contraste de pression. Il suppose des erreurs relatives sur les volumes. À incertitude absolue fixe, la précision relative se dégrade lorsque le compartiment rétrécit. Une erreur multiplicative identique sur les deux volumes s’annule exactement dans leur rapport.

## Niveau 3 — NUMERICAL-TEST

Plan fixé avant simulation. Pressions indépendantes uniformes entre 80 et 120 kPa, donc $D=40/\sqrt{6}$ kPa. Volume total nominal 10 L. Fractions cibles 50 %, 20 %, 5 % et 1 %. 50 000 tirages par scénario, avec graine fixe et tirages communs pour comparer les configurations. Bruit gaussien pour les pressions, uniforme borné pour les volumes afin de garantir des volumes mesurés positifs. Le programme calcule le poids mesuré à partir des deux volumes perturbés et utilise l’inversion exacte.

Trois scénarios pour chaque fraction : volumes exacts à la limite analytique de 1 kPa, allocation de 0,60 kPa aux pressions et de 0,60 kPa aux volumes (RMSE combinée d’environ 0,85 kPa), puis même bruit en pression avec deux fois plus d’incertitude volumique (environ 1,34 kPa). Critères : écart relatif simulation–théorie inférieur à 3 %, RMSE combinée sous 1 kPa pour le budget de travail et au-dessus pour le scénario dégradé. Vérifier aussi que l’erreur d’échelle commune des volumes s’annule.

In [1]:
from math import hypot, isclose, sqrt
from random import Random
from statistics import fmean

SEED = 20261121
N = 50_000
TARGET = 1.0
COMPONENT_BUDGET = 0.60
TOTAL_VOLUME = 10.0
FRACTIONS = (0.50, 0.20, 0.05, 0.01)
CONTRAST_RMS = 40 / sqrt(6)

def reconstruct(p1, pf, v1, v2):
    return p1 + (pf - p1) * (v1 + v2) / v2

assert isclose(reconstruct(90, 92, 9, 1), 110)
assert isclose(reconstruct(90, 92, 9*1.1, 1*1.1), 110)

rng = Random(SEED)
draws = [
    (rng.uniform(80, 120), rng.uniform(80, 120),
     rng.gauss(0, 1), rng.gauss(0, 1),
     rng.uniform(-sqrt(3), sqrt(3)), rng.uniform(-sqrt(3), sqrt(3)))
    for _ in range(N)
]

def simulate(w, pressure_sd, relative_volume_sd):
    v1, v2 = TOTAL_VOLUME * (1-w), TOTAL_VOLUME * w
    errors_squared = []
    for p1, p2, z1, zf, u1, u2 in draws:
        pf = p1 + w * (p2 - p1)
        estimate = reconstruct(
            p1 + pressure_sd*z1, pf + pressure_sd*zf,
            v1 * (1 + relative_volume_sd*u1),
            v2 * (1 + relative_volume_sd*u2),
        )
        errors_squared.append((estimate - p2)**2)
    return sqrt(fmean(errors_squared))

print(f"{N:,} tirages | graine {SEED}")
print("Part V2 | σp limite¹ | σp retenu | σV/V retenu | σV2 (mL) | RMSE limite¹ | RMSE budget | RMSE volume ×2")
for w in FRACTIONS:
    kp = hypot(1, 1-w) / w
    kv = sqrt(2) * (1-w) * CONTRAST_RMS
    pressure_limit = TARGET / kp
    pressure_sd = COMPONENT_BUDGET / kp
    relative_volume_sd = COMPONENT_BUDGET / kv
    scenarios = (
        (pressure_limit, 0),
        (pressure_sd, relative_volume_sd),
        (pressure_sd, 2*relative_volume_sd),
    )
    measured = []
    for sigma_p, r in scenarios:
        assert sqrt(3)*r < 1
        rmse = simulate(w, sigma_p, r)
        assert isclose(rmse, hypot(kp*sigma_p, kv*r), rel_tol=0.03)
        measured.append(rmse)
    exact_volumes, budget, degraded = measured
    assert budget < TARGET < degraded
    print(f"{w:7.0%} | {pressure_limit:10.5f} | {pressure_sd:9.5f} | "
          f"{relative_volume_sd:10.2%} | {1000*TOTAL_VOLUME*w*relative_volume_sd:8.2f} | "
          f"{exact_volumes:12.3f} | {budget:11.3f} | {degraded:15.3f}")
print("¹ Volumes exacts. Toutes les pressions et RMSE sont en kPa.")
print("PASS_SYNTHETIC_GAS_MEASUREMENT_ERROR_BUDGET")

50,000 tirages | graine 20261121
Part V2 | σp limite¹ | σp retenu | σV/V retenu | σV2 (mL) | RMSE limite¹ | RMSE budget | RMSE volume ×2
    50% |    0.44721 |   0.26833 |      5.20% |   259.81 |        0.999 |       0.853 |           1.378
    20% |    0.15617 |   0.09370 |      3.25% |    64.95 |        0.997 |       0.850 |           1.360
     5% |    0.03625 |   0.02175 |      2.73% |    13.67 |        0.997 |       0.849 |           1.356
     1% |    0.00711 |   0.00426 |      2.62% |     2.62 |        0.997 |       0.849 |           1.356
¹ Volumes exacts. Toutes les pressions et RMSE sont en kPa.
PASS_SYNTHETIC_GAS_MEASUREMENT_ERROR_BUDGET


## Niveau 4 — Verdict et portée

Le budget conjoint permet de dimensionner les mesures pour une RMSE cible dans ce modèle synthétique. Les chiffres du scénario retenu valident une moyenne quadratique sur les tirages. Pour garantir un budget linéarisé à un contraste donné, remplacer $D$ par ce contraste maximal, soit 40 kPa pour la plage testée. Des biais de capteur, des corrélations, des écarts de température ou une plage différente de pressions nécessitent un nouveau budget. Aucune donnée externe n’est requise.